# logit-thermo A2-6：データ集め（乳の種類、Qwen2.5（0.5B・1.5B・3B）、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 2 時間。終わったら自動で切れます。
- ゲート：`docs/A2_6_GATE.md`。この 1 本は Qwen2.5（0.5B・1.5B・3B） の 3 回。A2-6 のデータ集めは家族ごとに 3 本あり、同時に回して大丈夫です。
- 記事は A2-2 と同じ 40 本（A2-2 の種で選ぶ）。生成の種は A2-5 と同じ（20261102）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）と `scripts/a2_5_collect.py`（`19b523654033b24c…`）は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_6_collect.py`（`107229225bf9ac6f…`）は埋め込み。**logit-thermo フォルダの前のノートブックは動かさないでください。**
- 保存先 `A2_6/`。途中で切れても、もう一度「すべて実行」すれば終わった回は飛ばします。
- **通知**：🚀・回ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-6 データ集め（乳の種類、Qwen2.5（0.5B・1.5B・3B）、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-6 データ集め（乳の種類、Qwen2.5（0.5B・1.5B・3B）、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-6 collect qwen25', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC5C': '19b523654033b24ce50ede75922d4a5ceeff0dbfa59b8a197d4920e1b19db8c8'})
SRC22, SRC5C = _s['SRC22'], _s['SRC5C']
SRC6C = '"""A2-6（ゲート docs/A2_6_GATE.md）のデータ集め：「乳の種類」テスト。GPU（L4）のノートブックで、\nscripts/a2_2_collect.py と scripts/a2_5_collect.py を実行したのと同じ名前空間で実行する。\nA2-2 の run_model・generate と、A2-5 の make_chunked・pairs_hash をそのまま使う。\n- 生成のあいだだけ SEED を A2-5 と同じ 20261102 にする（Pythia の 6 回は A2-5 のデータをそのまま使うため）。\n- 家族（グループ）ごとに 1 つのノートブックで回す。家族の中でトークナイザーの区切り方が同じかを確かめる。\n"""\nimport json, time\nimport numpy as np\n\nA6_SEED = 20261102\nA6_GROUPS = {  # (名前, Hugging Face の ID, 生成を分ける本数（None なら分けない）)\n    \'qwen25\': [(\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\', None), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', None),\n               (\'qwen2.5-3b\', \'Qwen/Qwen2.5-3B\', 20)],\n    \'smol2\': [(\'smollm2-135m\', \'HuggingFaceTB/SmolLM2-135M\', None), (\'smollm2-360m\', \'HuggingFaceTB/SmolLM2-360M\', None),\n              (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', None)],\n    \'qwen3\': [(\'qwen3-0.6b\', \'Qwen/Qwen3-0.6B-Base\', None), (\'qwen3-1.7b\', \'Qwen/Qwen3-1.7B-Base\', None),\n              (\'qwen3-4b\', \'Qwen/Qwen3-4B-Base\', 20)],\n}\n\n\ndef hub_sha(hf_id):\n    """いまの main の版（コミット）を記録する。transformers 5 では config の _commit_hash が空になるため。"""\n    try:\n        from huggingface_hub import HfApi\n        return HfApi().model_info(hf_id).sha\n    except Exception as e:\n        return f\'取れませんでした：{type(e).__name__}\'\n\n\ndef run_a26(group, docs, outdir, device, notify=print):\n    """docs は先に load_docs()（A2-2 の種 20261027）で読んでおくこと。"""\n    global SEED, generate\n    from transformers import AutoTokenizer\n    keep_seed, keep_gen = SEED, generate\n    runs = A6_GROUPS[group]\n    hs = {name: pairs_hash(AutoTokenizer.from_pretrained(hf_id), docs) for name, hf_id, _ in runs}\n    assert len(set(hs.values())) == 1, (\'家族の中でトークナイザーの区切り方が違います\', hs)\n    meta = {\'group\': group, \'gen_seed\': A6_SEED, \'doc_seed\': keep_seed, \'pairs_sha256\': list(hs.values())[0], \'runs\': {}}\n    try:\n        for name, hf_id, chunk in runs:\n            t0 = time.time()\n            SEED = A6_SEED\n            generate = make_chunked(keep_gen, chunk) if chunk else keep_gen\n            assert run_model.__globals__[\'SEED\'] == A6_SEED and run_model.__globals__[\'generate\'] is generate\n            try:\n                run_model(name, hf_id, docs, outdir, device, notify=notify)\n            finally:\n                SEED, generate = keep_seed, keep_gen\n            Z = np.load(f\'{outdir}/{name}.npz\')\n            meta[\'runs\'][name] = {\'hf_id\': hf_id, \'chunk\': chunk, \'use\': [int(u) for u in Z[\'use\']],\n                                  \'hub_sha\': hub_sha(hf_id), \'seconds\': round(time.time() - t0, 1)}\n            json.dump(meta, open(f\'{outdir}/a2_6_runs_{group}.json\', \'w\'), ensure_ascii=False, indent=1)\n    finally:\n        SEED, generate = keep_seed, keep_gen\n    return meta\n\n\ndef self_test_a26():\n    names = [n for g in A6_GROUPS.values() for n, _, _ in g]\n    assert len(set(names)) == len(names)\n    assert not any(n.startswith(\'pythia\') for n in names)\n    assert \'SEED\' in run_model.__globals__ and \'generate\' in run_model.__globals__\n    return \'OK\'\n'
assert hashlib.sha256(SRC6C.encode()).hexdigest() == '107229225bf9ac6f56e639912392dd17ed495364b6ed3b1554b08d426a53efbd'
exec(SRC22)
exec(SRC5C)
exec(SRC6C)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 の種で選び、A2-2 と同じか確かめる）→ 3 回 → 保存（結果は表示しない）
import json, sys, transformers
GROUP = 'qwen25'
try:
    print('自己テスト：', self_test('cuda'), self_test_a26())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_6', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    m = run_a26(GROUP, docs, '/content/drive/MyDrive/logit-thermo/A2_6', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    assert SEED == 20261027, '種が戻っていません'
    m.update(gate='docs/A2_6_GATE.md', a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', a2_5_collect_sha256='19b523654033b24ce50ede75922d4a5ceeff0dbfa59b8a197d4920e1b19db8c8', a2_6_collect_sha256='107229225bf9ac6f56e639912392dd17ed495364b6ed3b1554b08d426a53efbd',
             torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
             gpu=torch.cuda.get_device_name(0), float32_matmul_precision=torch.get_float32_matmul_precision(),
             max_memory_allocated_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    json.dump(m, open('/content/drive/MyDrive/logit-thermo/A2_6' + f'/run_meta_{GROUP}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-6 のデータ集め（Qwen2.5（0.5B・1.5B・3B））で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-6 の Qwen2.5（0.5B・1.5B・3B） を A2_6 に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
